In [116]:
import requests
import os
from dotenv import load_dotenv
import xml.etree.ElementTree as ET
import pandas as pd
from datetime import datetime, time
from zoneinfo import ZoneInfo

In [117]:
load_dotenv()

True

In [118]:
api_key = os.getenv("ENTSOE_API_KEY")

In [119]:
payload={}
headers = {
  "SECURITY_TOKEN": api_key
}

In [120]:
def convert_local_midnight_to_utc(date_str):
    pure_date = datetime.strptime(date_str, "%Y-%m-%d").date()
    prague_tz = ZoneInfo("Europe/Prague")
    local_midnight = datetime.combine(pure_date, time.min, tzinfo=prague_tz)
    utc_dt = local_midnight.astimezone(ZoneInfo("UTC"))
    
    # ENTSO-E needs YYYYMMDDHHmm
    return utc_dt.strftime("%Y%m%d%H%M")

In [121]:
data = []
df_hourly = pd.DataFrame()

def parse_xml_response(data, xml_text, namespace):
    root = ET.fromstring(response.text)
    ns = {"ns": namespace}
    
    for period_block in root.findall(".//ns:Period", ns):
        day = period_block.find("ns:timeInterval/ns:start", ns).text
        
        for point in period_block.findall("ns:Point", ns):
            pos = float(point.find("ns:position", ns).text)
            quantity = float(point.find("ns:quantity", ns).text)
            
            data.append({"pos": pos, "load": quantity})

In [122]:
# data for 2023
url = f"https://web-api.tp.entsoe.eu/api?documentType=A65&processType=A01&outBiddingZone_Domain=10YCZ-CEPS-----N&periodStart={convert_local_midnight_to_utc('2023-01-01')}&periodEnd={convert_local_midnight_to_utc('2024-01-01')}"
response = requests.request("GET", url, headers=headers, data=payload)

parse_xml_response(data, response.text, "urn:iec62325.351:tc57wg16:451-6:generationloaddocument:3:0")

df_temp = pd.DataFrame(data)
df_temp["pos"] = df_temp["pos"].astype(int)
df_temp = (
    df_temp.set_index("pos")
    .reindex(range(1, df_temp["pos"].max() + 1))
    .ffill()
    .reset_index()
)

print(len(data))
print(len(df_temp))
    
df_hourly = pd.concat([df_hourly, df_temp], ignore_index=True)
data = []

8745
8760


In [123]:
# data for 2024 until 2024-07-01
url = f"https://web-api.tp.entsoe.eu/api?documentType=A65&processType=A01&outBiddingZone_Domain=10YCZ-CEPS-----N&periodStart={convert_local_midnight_to_utc('2024-01-01')}&periodEnd={convert_local_midnight_to_utc('2024-06-30')}"
response = requests.request("GET", url, headers=headers, data=payload)

parse_xml_response(data, response.text, "urn:iec62325.351:tc57wg16:451-6:generationloaddocument:3:0")

df_temp = pd.DataFrame(data)
df_temp["pos"] = df_temp["pos"].astype(int)
df_temp = (
    df_temp.set_index("pos")
    .reindex(range(1, df_temp["pos"].max() + 1))
    .ffill()
    .reset_index()
)

print(len(data))
print(len(df_temp))
    
df_hourly = pd.concat([df_hourly, df_temp], ignore_index=True)
data = []

4336
4343


In [124]:
# Entso-e struggles to return consistent data for this date (it is last day with hour period, 2024-07-01 is first day with 15min period)
data = [
  {
    "pos": 1,
    "load": 4949.0
  },
  {
    "pos": 2,
    "load": 4730.0
  },
  {
    "pos": 3,
    "load": 4618.0
  },
  {
    "pos": 4,
    "load": 4576.0
  },
  {
    "pos": 5,
    "load": 4631.0
  },
  {
    "pos": 6,
    "load": 4502.0
  },
  {
    "pos": 7,
    "load": 4558.0
  },
  {
    "pos": 8,
    "load": 5006.0
  },
  {
    "pos": 9,
    "load": 5598.0
  },
  {
    "pos": 10,
    "load": 6195.0
  },
  {
    "pos": 11,
    "load": 6596.0
  },
  {
    "pos": 12,
    "load": 6899.0
  },
  {
    "pos": 13,
    "load": 6857.0
  },
  {
    "pos": 14,
    "load": 6727.0
  },
  {
    "pos": 15,
    "load": 6552.0
  },
  {
    "pos": 16,
    "load": 6461.0
  },
  {
    "pos": 17,
    "load": 6382.0
  },
  {
    "pos": 18,
    "load": 6157.0
  },
  {
    "pos": 19,
    "load": 6039.0
  },
  {
    "pos": 20,
    "load": 6033.0
  },
  {
    "pos": 21,
    "load": 5933.0
  },
  {
    "pos": 22,
    "load": 5897.0
  },
  {
    "pos": 23,
    "load": 5906.0
  },
  {
    "pos": 24,
    "load": 5656.0
  }
]

df_temp = pd.DataFrame(data)
df_temp["pos"] = df_temp["pos"].astype(int)
df_temp = (
    df_temp.set_index("pos")
    .reindex(range(1, df_temp["pos"].max() + 1))
    .ffill()
    .reset_index()
)

print(len(data))
print(len(df_temp))
    
df_hourly = pd.concat([df_hourly, df_temp], ignore_index=True)
data = []

24
24


In [125]:
df_quarterly = df_hourly.loc[df_hourly.index.repeat(4)].reset_index(drop=True)
print(len(df_quarterly))

52508


In [126]:
# data for the rest of 2024
url = f"https://web-api.tp.entsoe.eu/api?documentType=A65&processType=A01&outBiddingZone_Domain=10YCZ-CEPS-----N&periodStart={convert_local_midnight_to_utc('2024-07-01')}&periodEnd={convert_local_midnight_to_utc('2025-01-01')}"
response = requests.request("GET", url, headers=headers, data=payload)

parse_xml_response(data, response.text, "urn:iec62325.351:tc57wg16:451-6:generationloaddocument:3:0")

df_temp = pd.DataFrame(data)
df_temp["pos"] = df_temp["pos"].astype(int)
df_temp = (
    df_temp.set_index("pos")
    .reindex(range(1, df_temp["pos"].max() + 1))
    .ffill()
    .reset_index()
)

print(len(data))
print(len(df_temp))
    
df_quarterly = pd.concat([df_quarterly, df_temp], ignore_index=True)
data = []

17549
17668


In [127]:
# data 2025
url = f"https://web-api.tp.entsoe.eu/api?documentType=A65&processType=A01&outBiddingZone_Domain=10YCZ-CEPS-----N&periodStart={convert_local_midnight_to_utc('2025-01-01')}&periodEnd={convert_local_midnight_to_utc('2026-01-01')}"
response = requests.request("GET", url, headers=headers, data=payload)

parse_xml_response(data, response.text, "urn:iec62325.351:tc57wg16:451-6:generationloaddocument:3:0")

df_temp = pd.DataFrame(data)
df_temp["pos"] = df_temp["pos"].astype(int)
df_temp = (
    df_temp.set_index("pos")
    .reindex(range(1, df_temp["pos"].max() + 1))
    .ffill()
    .reset_index()
)

print(len(data))
print(len(df_temp))
    
df_quarterly = pd.concat([df_quarterly, df_temp], ignore_index=True)
data = []

34850
35040


In [128]:
df_quarterly["load_prediction"] = df_quarterly["load"]
df_quarterly = df_quarterly.drop(columns=["load", "pos"])

In [129]:
df_quarterly.to_csv("../data/czechia_load_prediction_15min.csv", sep=";", index=False)